# Smart Attendance System Notebook



## 1. Install and Verify Dependencies

If a package is missing, install it in a notebook cell before running the rest of the notebook.

In [1]:
import importlib.util

packages = ["cv2", "numpy", "face_recognition"]
for package_name in packages:
    available = importlib.util.find_spec(package_name) is not None
    print(f"{package_name}: {'OK' if available else 'MISSING'}")

cv2: OK
numpy: OK
face_recognition: OK


## 2. Import Libraries and Define Paths

Import the required libraries and define shared file paths.

In [2]:
import csv
import os
from datetime import datetime

import cv2
import numpy as np
import face_recognition

path = 'images'
images = []
classNames = []
attendance_file = 'attendance.csv'

## 3. Load Known Face Images and Labels

Read every image from the `images` folder and store the file-name stems as labels.

In [3]:
myList = os.listdir(path)

for cl in myList:
    curImg = cv2.imread(f'{path}/{cl}')
    images.append(curImg)
    classNames.append(os.path.splitext(cl)[0])

print(f'Loaded {len(images)} images')

Loaded 3 images


## 4. Compute Face Encodings for Known Images

Build face encodings for the known reference images.

In [4]:
def findEncodings(images):
    encodeList = []
    for img in images:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        encode = face_recognition.face_encodings(img)[0]
        encodeList.append(encode)
    return encodeList

encodeListKnown = findEncodings(images)
print('Encoding Complete')

Encoding Complete


## 5. Initialize Attendance CSV and Logging Function

Create the attendance file handler and avoid duplicate same-day entries.

In [5]:
def markAttendance(name):
    now = datetime.now()
    date = now.strftime('%Y-%m-%d')
    time = now.strftime('%H:%M:%S')

    with open(attendance_file, 'a+', newline='') as f:
        f.seek(0)
        reader = csv.reader(f)
        data = list(reader)

        already_marked = False

        for row in data:
            if len(row) > 0 and row[0] == name and len(row) > 1 and row[1] == date:
                already_marked = True
                break

        if not already_marked:
            writer = csv.writer(f)
            writer.writerow([name, date, time])

## 6. Start Webcam Capture

Open the webcam and confirm that the camera is available.

In [6]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError('Could not open the webcam.')

print('Webcam ready')

Webcam ready


## 7. Run Real-Time Face Recognition Loop

Read webcam frames, compare them against the known encodings, and log attendance.

In [7]:
while True:
    success, img = cap.read()

    if not success:
        continue

    imgS = cv2.resize(img, (0, 0), None, 0.25, 0.25)
    imgS = cv2.cvtColor(imgS, cv2.COLOR_BGR2RGB)

    facesCurFrame = face_recognition.face_locations(imgS)
    encodesCurFrame = face_recognition.face_encodings(imgS, facesCurFrame)

    for encodeFace, faceLoc in zip(encodesCurFrame, facesCurFrame):
        matches = face_recognition.compare_faces(encodeListKnown, encodeFace)
        faceDis = face_recognition.face_distance(encodeListKnown, encodeFace)

        matchIndex = np.argmin(faceDis)

        if matches[matchIndex]:
            name = classNames[matchIndex].upper()

            y1, x2, y2, x1 = faceLoc
            y1, x2, y2, x1 = y1 * 4, x2 * 4, y2 * 4, x1 * 4

            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(img, name, (x1, y2 + 20), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
            markAttendance(name)

    cv2.imshow('Recognition', img)

    if cv2.waitKey(1) == ord('q'):
        break

## 8. Release Camera and Close OpenCV Windows

Run this cleanup cell after stopping the loop to free the webcam and close windows.

In [8]:
cap.release()
cv2.destroyAllWindows()
print('Resources released')

Resources released
